# Data Leakage Prevention During Preprocessing

## Definition
**Data leakage** occurs when information from outside the training dataset is used to build the model,
leading to overly optimistic performance metrics that don't generalize.

### Types of Leakage
| Type | Description | Example |
|------|-------------|---------|
| **Target leakage** | Feature contains future/outcome info | Using `days_late` to predict loan default |
| **Train-test leakage** | Test data influences training | Scaling with full-dataset mean |
| **Temporal leakage** | Future data used to predict past | Using future rolling average as feature |
| **Row leakage** | Same entity in train and test | Same customer in both splits |
| **Preprocessing leakage** | Stats computed before split | Imputing with global mean before split |

### Rules to Prevent Leakage
1. **Split FIRST, then preprocess** — never compute statistics on the full dataset
2. **Use sklearn Pipelines** — fit only on train, transform both
3. **Remove target-correlated features** from training
4. **Sort by time** for temporal splits
5. **Group split** — same entity (user, patient) only in one split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(42)
n = 1000

# Dataset: predict if customer churns
df = pd.DataFrame({
    'age':    rng.integers(18, 65, n),
    'income': rng.normal(60000, 20000, n),
    'score':  rng.normal(700, 50, n),
    'churn':  rng.choice([0, 1], n, p=[0.8, 0.2])
})
print(df.shape)

In [ ]:
# BAD: Scale before split (LEAKAGE!)
from sklearn.model_selection import train_test_split

X = df[['age', 'income', 'score']]
y = df['churn']

# WRONG: fitting scaler on ALL data
scaler_bad = StandardScaler()
X_scaled_bad = scaler_bad.fit_transform(X)   # LEAKS test statistics!

X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_scaled_bad, y, test_size=0.2, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_train_bad, y_train)
print(f'Leaky accuracy: {model.score(X_test_bad, y_test):.4f}')

In [ ]:
# GOOD: Split FIRST, then scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler ONLY on train
scaler_good = StandardScaler()
X_train_scaled = scaler_good.fit_transform(X_train)   # fit on train
X_test_scaled  = scaler_good.transform(X_test)         # only transform test!

model.fit(X_train_scaled, y_train)
print(f'Non-leaky accuracy: {model.score(X_test_scaled, y_test):.4f}')

In [ ]:
# BEST: Use sklearn Pipeline (guarantees no leakage)
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

# Cross-validation with pipeline — scaler is fit fresh for each fold
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f'Pipeline CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# Target Encoding Leakage
rng2 = np.random.default_rng(42)
df_cat = pd.DataFrame({
    'city': rng2.choice(['NYC', 'LA', 'Chicago'], n),
    'target': rng2.choice([0, 1], n, p=[0.7, 0.3])
})

# BAD: compute encoding on full data
leaky_enc = df_cat.groupby('city')['target'].mean()
df_cat['city_enc_bad'] = df_cat['city'].map(leaky_enc)
print('Leaky encoding (correlation with target):')
print(df_cat[['city_enc_bad', 'target']].corr()['target']['city_enc_bad'].round(4))

# GOOD: compute encoding only on train, apply to test
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    df_cat[['city']], df_cat['target'], test_size=0.2, random_state=42)

train_with_target = X_train_c.copy()
train_with_target['target'] = y_train_c.values
enc_map = train_with_target.groupby('city')['target'].mean()

X_train_c['city_enc'] = X_train_c['city'].map(enc_map)
X_test_c['city_enc']  = X_test_c['city'].map(enc_map).fillna(y_train_c.mean())
print('Non-leaky encoding — train encoding values:')
print(enc_map.round(4))

In [ ]:
# Imputation Leakage
df_imp = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5, 2, np.nan, 4],
    'B': [10, 20, np.nan, 40, 50, np.nan, 70, 80],
    'target': [0, 1, 0, 1, 0, 0, 1, 0]
})

X = df_imp[['A', 'B']]
y = df_imp['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# BAD: impute with global mean (leaks test stats)
global_mean = X.mean()  # uses ALL data including test!
print('Leaky imputation mean (from ALL data):', global_mean.round(3).to_dict())

# GOOD: impute with TRAIN mean only
train_mean = X_train.mean()  # only training data
X_train_imp = X_train.fillna(train_mean)
X_test_imp  = X_test.fillna(train_mean)   # apply train stats to test
print('Correct imputation mean (from TRAIN):', train_mean.round(3).to_dict())